In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightDelayPredictionBigData") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "100s") \
    .config("spark.rdd.compress", "true") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

HDFS_PATH = "hdfs://localhost:9000/data/"

In [2]:
airlines = spark.read.csv(HDFS_PATH + "airlines.csv", header=True, inferSchema=True, sep = ",")
airlines.show(2)

+---------+--------------------+
|IATA_CODE|             AIRLINE|
+---------+--------------------+
|       UA|United Air Lines ...|
|       AA|American Airlines...|
+---------+--------------------+
only showing top 2 rows


In [3]:
airports = spark.read.csv(HDFS_PATH + "airports.csv", header=True, inferSchema=True, sep = ",")
airports.show(2)

+---------+--------------------+---------+-----+-------+--------+---------+
|IATA_CODE|             AIRPORT|     CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+--------------------+---------+-----+-------+--------+---------+
|      ABE|Lehigh Valley Int...|Allentown|   PA|    USA|40.65236| -75.4404|
|      ABI|Abilene Regional ...|  Abilene|   TX|    USA|32.41132| -99.6819|
+---------+--------------------+---------+-----+-------+--------+---------+
only showing top 2 rows


In [4]:
flights = spark.read.csv(HDFS_PATH + "flights.csv", header=True, inferSchema=True, sep = ",")
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

**FORMATING TIME TYPE FOR SCHEDULED_DEPARTURE, DEPARTURE_TIME, SCHEDULED_ARRIVAL, ARRIVAL_TIME**

# Example: 830 --> 08:30

In [5]:
from pyspark.sql.functions import col, when, lpad, concat, lit, substring

TIME_COLS = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME']

for tcol in TIME_COLS:
    time_string_col = lpad(
        when(col(tcol) == 2400, 0)
        .otherwise(col(tcol))
        .cast("int").cast("string"),
        4, "0"
    )

    flights = flights.withColumn(
        tcol,
        concat(substring(time_string_col, 1, 2), lit(":"), substring(time_string_col, 3, 2))
    )


# Creating label --> Delay > 15 minutes --> 1 (Delay), otherwise 0 (In time)

In [6]:
from pyspark.sql import functions as F

flights = flights.filter(F.col("CANCELLED") == 0)

flights = flights.withColumn(
    "label",
    F.when(F.col("ARRIVAL_DELAY") > 15, 1).otherwise(0)
)

# Example 08:30 --> 8*60 + 30

In [7]:
flights = flights.withColumn(
    "DEP_TIME_MINUTES",
    F.split(F.col("DEPARTURE_TIME"), ":")[0].cast("int") * 60 +
    F.split(F.col("DEPARTURE_TIME"), ":")[1].cast("int")
)

In [8]:
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+----------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|label|DEP_TIME_MINUTES|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+---

# Pipeline String, OneHot for categorical cols --> assembling vector --> model (Logistic Regression)

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages.append(assembler)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
    # maxIter=20,
    # regParam=0.1,
    # elasticNetParam=0.0
)
stages.append(lr)

pipeline = Pipeline(stages=stages)

# Splitting train vs test --> creating classWeight --> predicting

In [10]:
train_data, test_data = flights.randomSplit([0.8, 0.2], seed=42)

total_count = train_data.count()
delay_count = train_data.filter(F.col("label") == 1.0).count()
on_time_count = total_count - delay_count

weight_for_delay = total_count / (2.0 * delay_count)
weight_for_ontime = total_count / (2.0 * on_time_count)

train_data_weighted = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1.0, weight_for_delay).otherwise(weight_for_ontime)
)

model = pipeline.fit(train_data_weighted)

predictions = model.transform(test_data)

In [11]:
model_path = "model/flight_logistic_model"
model.write().overwrite().save(model_path)

# Showing the results

In [12]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9363
2. Accuracy:      0.9075
3. Precision:     0.9156
4. Recall:        0.9075
5. F1-Score:      0.9103


In [13]:
from sklearn.metrics import classification_report
y_compare = predictions.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare["label"], y_compare["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    941207
       Delay       0.70      0.83      0.76    204092

    accuracy                           0.91   1145299
   macro avg       0.83      0.88      0.85   1145299
weighted avg       0.92      0.91      0.91   1145299



# Extracting and showing the importance of features

In [14]:
import pandas as pd
import numpy as np

lr_model = model.stages[-1]
coefficients = lr_model.coefficients.toArray()

features_metadata = predictions.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===
                     Feature_Name  Coefficient  Absolute_Importance
0   DESTINATION_AIRPORT_Vec_13459   -10.529381            10.529381
1   DESTINATION_AIRPORT_Vec_11503   -10.112292            10.112292
2   DESTINATION_AIRPORT_Vec_13541   -10.060583            10.060583
3   DESTINATION_AIRPORT_Vec_10666    -9.656471             9.656471
4        ORIGIN_AIRPORT_Vec_14960    -9.333105             9.333105
5        ORIGIN_AIRPORT_Vec_14006    -9.129696             9.129696
6   DESTINATION_AIRPORT_Vec_12016    -9.129185             9.129185
7   DESTINATION_AIRPORT_Vec_13127    -9.093438             9.093438
8        ORIGIN_AIRPORT_Vec_10268    -8.919617             8.919617
9        ORIGIN_AIRPORT_Vec_13502    -8.560259             8.560259
10       ORIGIN_AIRPORT_Vec_11503    -8.373560             8.373560
11       ORIGIN_AIRPORT_Vec_14150    -6.235228             6.235228
12  DESTINATION_AIRPORT_Vec_11315    -4.791955         

### Decision Tree

In [15]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages_dt = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_dt += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_dt = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_dt.append(assembler_dt)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm
    maxDepth=5,
    maxBins=32,
    impurity="gini"
)
stages_dt.append(dt)

pipeline_dt = Pipeline(stages=stages_dt)

In [16]:
model_dt = pipeline_dt.fit(train_data_weighted)

predictions_dt = model_dt.transform(test_data)

In [17]:
model_dt_path = "model/flight_decision_tree_model"
model_dt.write().overwrite().save(model_dt_path)

In [18]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_dt)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_dt)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_dt)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_dt)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_dt)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.7674
2. Accuracy:      0.9011
3. Precision:     0.9121
4. Recall:        0.9011
5. F1-Score:      0.9048


In [19]:
y_compare_dt = predictions_dt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_dt["label"], y_compare_dt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    941207
       Delay       0.68      0.83      0.75    204092

    accuracy                           0.90   1145299
   macro avg       0.82      0.87      0.84   1145299
weighted avg       0.91      0.90      0.90   1145299



In [35]:
dt_model = model_dt.stages[-1]
feature_importances = dt_model.featureImportances.toArray()

features_metadata = predictions_dt.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (DECISION TREE) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (DECISION TREE) ===
                     Feature_Name  Importance
0                 DEPARTURE_DELAY    0.997948
1                  AIRLINE_Vec_WN    0.001324
2                  AIRLINE_Vec_UA    0.000644
3     DESTINATION_AIRPORT_Vec_LAX    0.000085
4   DESTINATION_AIRPORT_Vec_12264    0.000000
5     DESTINATION_AIRPORT_Vec_GNV    0.000000
6     DESTINATION_AIRPORT_Vec_ISN    0.000000
7   DESTINATION_AIRPORT_Vec_14492    0.000000
8   DESTINATION_AIRPORT_Vec_14683    0.000000
9     DESTINATION_AIRPORT_Vec_BFL    0.000000
10    DESTINATION_AIRPORT_Vec_LNK    0.000000
11    DESTINATION_AIRPORT_Vec_BMI    0.000000
12    DESTINATION_AIRPORT_Vec_TVC    0.000000
13    DESTINATION_AIRPORT_Vec_BTV    0.000000
14    DESTINATION_AIRPORT_Vec_MFR    0.000000


### GBT

In [21]:
from pyspark.ml.classification import GBTClassifier

stages_gbt = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_gbt += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_gbt = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_gbt.append(assembler_gbt)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm
    maxIter=10,
    maxDepth=4,
    stepSize=0.1
)
stages_gbt.append(gbt)

pipeline_gbt = Pipeline(stages=stages_gbt)

In [22]:
model_gbt = pipeline_gbt.fit(train_data_weighted)

predictions_gbt = model_gbt.transform(test_data)

In [23]:
model_gbt_path = "model/flight_gbt_model"
model_gbt.write().overwrite().save(model_gbt_path)

In [24]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_gbt)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_gbt)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_gbt)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_gbt)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_gbt)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9326
2. Accuracy:      0.9026
3. Precision:     0.9130
4. Recall:        0.9026
5. F1-Score:      0.9061


In [25]:
y_compare_gbt = predictions_gbt.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_gbt["label"], y_compare_gbt["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    941207
       Delay       0.69      0.83      0.75    204092

    accuracy                           0.90   1145299
   macro avg       0.82      0.88      0.85   1145299
weighted avg       0.91      0.90      0.91   1145299



In [26]:
gbt_model = model_gbt.stages[-1]
feature_importances = gbt_model.featureImportances.toArray()

features_metadata = predictions_gbt.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (GBT) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (GBT) ===
                   Feature_Name  Importance
0               DEPARTURE_DELAY    0.939546
1                AIRLINE_Vec_DL    0.012761
2                AIRLINE_Vec_WN    0.011707
3                      DISTANCE    0.011095
4                AIRLINE_Vec_UA    0.010685
5   DESTINATION_AIRPORT_Vec_LAX    0.002868
6   DESTINATION_AIRPORT_Vec_ORD    0.002258
7                   MONTH_Vec_2    0.001816
8        ORIGIN_AIRPORT_Vec_ATL    0.001629
9                AIRLINE_Vec_HA    0.001166
10  DESTINATION_AIRPORT_Vec_HNL    0.000974
11               AIRLINE_Vec_US    0.000935
12       ORIGIN_AIRPORT_Vec_LGA    0.000771
13  DESTINATION_AIRPORT_Vec_LGA    0.000702
14                  MONTH_Vec_1    0.000560


### Random Forest

In [27]:
# Tính ratio
total = train_data.count()
n_delay = train_data.filter(F.col("label") == 1).count()
n_ontime = train_data.filter(F.col("label") == 0).count()

weight_delay  = total / (2 * n_delay)
weight_ontime = total / (2 * n_ontime)

train_data_w = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, weight_delay).otherwise(weight_ontime)
)

In [28]:
from pyspark.ml.classification import RandomForestClassifier

stages_rf = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_rf += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_rf = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_rf.append(assembler_rf)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",   # thêm dòng này
    numTrees=100,
    maxDepth=5,
    seed=42
)
stages_rf.append(rf)

pipeline_rf = Pipeline(stages=stages_rf)

In [29]:
model_rf = pipeline_rf.fit(train_data_weighted)  # dùng train_data_w thay vì train_data
predictions_rf = model_rf.transform(test_data)

In [30]:
model_rf_path = "model/flight_random_forest_model"
model_rf.write().overwrite().save(model_rf_path)

In [31]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_rf)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_rf)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_rf)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_rf)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_rf)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.8894
2. Accuracy:      0.9127
3. Precision:     0.9162
4. Recall:        0.9127
5. F1-Score:      0.9141


In [32]:
y_compare_rf = predictions_rf.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_rf["label"], y_compare_rf["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.94      0.95    941207
       Delay       0.73      0.80      0.77    204092

    accuracy                           0.91   1145299
   macro avg       0.85      0.87      0.86   1145299
weighted avg       0.92      0.91      0.91   1145299



In [33]:
rf_model = model_rf.stages[-1]
feature_importances = rf_model.featureImportances.toArray()

features_metadata = predictions_rf.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Importance': feature_importances
})

feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (RANDOM FOREST) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (RANDOM FOREST) ===
                     Feature_Name  Importance
0                 DEPARTURE_DELAY    0.208203
1                DEP_TIME_MINUTES    0.142107
2                  AIRLINE_Vec_DL    0.063441
3                  AIRLINE_Vec_NK    0.060582
4                     MONTH_Vec_2    0.042448
5                     MONTH_Vec_6    0.032774
6          ORIGIN_AIRPORT_Vec_ORD    0.029282
7                     MONTH_Vec_9    0.028957
8                    MONTH_Vec_11    0.024674
9               DAY_OF_WEEK_Vec_6    0.018524
10  DESTINATION_AIRPORT_Vec_10397    0.017106
11                 AIRLINE_Vec_B6    0.015842
12                 AIRLINE_Vec_F9    0.015490
13                    MONTH_Vec_7    0.015089
14    DESTINATION_AIRPORT_Vec_SLC    0.014688


### Linear SVM

In [36]:
from pyspark.ml.classification import LinearSVC

stages_svm = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages_svm += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler_svm = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_svm.append(assembler_svm)

svm = LinearSVC(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=20,
    regParam=0.1
)
stages_svm.append(svm)

pipeline_svm = Pipeline(stages=stages_svm)

In [37]:
model_svm = pipeline_svm.fit(train_data_weighted)
predictions_svm = model_svm.transform(test_data)

In [38]:
model_svm_path = "model/flight_svm_model"
model_svm.write().overwrite().save(model_svm_path)

In [39]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_svm)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_svm)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_svm)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_svm)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_svm)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9251
2. Accuracy:      0.9285
3. Precision:     0.9278
4. Recall:        0.9285
5. F1-Score:      0.9237


In [40]:
y_compare_svm = predictions_svm.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_svm["label"], y_compare_svm["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.93      0.99      0.96    941207
       Delay       0.92      0.66      0.77    204092

    accuracy                           0.93   1145299
   macro avg       0.92      0.82      0.86   1145299
weighted avg       0.93      0.93      0.92   1145299



In [41]:
svm_model = model_svm.stages[-1]
coefficients = svm_model.coefficients.toArray()

features_metadata = predictions_svm.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LINEAR SVM) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LINEAR SVM) ===
                     Feature_Name  Coefficient  Absolute_Importance
0        ORIGIN_AIRPORT_Vec_13964     1.164705             1.164705
1        ORIGIN_AIRPORT_Vec_14222     1.028976             1.028976
2          ORIGIN_AIRPORT_Vec_ADK     0.995720             0.995720
3          ORIGIN_AIRPORT_Vec_GST     0.982596             0.982596
4        ORIGIN_AIRPORT_Vec_15497     0.963609             0.963609
5   DESTINATION_AIRPORT_Vec_13964     0.952021             0.952021
6        ORIGIN_AIRPORT_Vec_10165     0.895629             0.895629
7          ORIGIN_AIRPORT_Vec_PPG     0.766168             0.766168
8   DESTINATION_AIRPORT_Vec_12016    -0.749576             0.749576
9        ORIGIN_AIRPORT_Vec_10154     0.729248             0.729248
10    DESTINATION_AIRPORT_Vec_GUM     0.713920             0.713920
11       ORIGIN_AIRPORT_Vec_13541     0.665345             0.665345
12  DESTINATION_AIRPORT_Vec_13502     0.665345             0.665

### Naive Bayes

In [42]:
from pyspark.ml.classification import NaiveBayes

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DISTANCE", "DEPARTURE_DELAY"]  # bỏ DEP_TIME_MINUTES vì có thể âm

stages_nb = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    stages_nb.append(indexer)

assembler_inputs = [col + "_Index" for col in categorical_cols] + numeric_cols
assembler_nb = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages_nb.append(assembler_nb)

nb = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    smoothing=1.0,
    modelType="gaussian"  # dùng gaussian vì features là continuous
)
stages_nb.append(nb)

pipeline_nb = Pipeline(stages=stages_nb)

In [43]:
model_nb = pipeline_nb.fit(train_data)
predictions_nb = model_nb.transform(test_data)

In [44]:
model_nb_path = "model/flight_naive_bayes_model"
model_nb.write().overwrite().save(model_nb_path)

In [45]:
print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions_nb)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions_nb)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions_nb)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions_nb)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions_nb)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.5277
2. Accuracy:      0.9333
3. Precision:     0.9315
4. Recall:        0.9333
5. F1-Score:      0.9303


In [46]:
y_compare_nb = predictions_nb.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare_nb["label"], y_compare_nb["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.94      0.98      0.96    941207
       Delay       0.89      0.71      0.79    204092

    accuracy                           0.93   1145299
   macro avg       0.92      0.85      0.88   1145299
weighted avg       0.93      0.93      0.93   1145299

